# Molab (Marimo) Master Orchestrator
This notebook is specifically configured for Molab (Marimo) where you manually upload the `Paper 1` directory to `/marimo/Paper 1`.

In [ ]:
!pip install -q papermill tabulate ipykernel

import subprocess, sys, ipykernel
# Register the current Python interpreter as the 'python3' kernel for Papermill
result = subprocess.run(
    [sys.executable, '-m', 'ipykernel', 'install', '--user', '--name', 'python3', '--display-name', 'Python 3'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)
print('Kernel registration complete.')

In [ ]:
import os
import glob
from tqdm.notebook import tqdm
import shutil

# 1. Find all model notebooks inside the exact absolute path
paper_dir = '/marimo/Paper 1'

if not os.path.exists(paper_dir):
    print(f'Error: Could not find directory at {paper_dir}. Please upload the folder correctly.')
    
all_notebooks = glob.glob(f'{paper_dir}/**/*.ipynb', recursive=True)
all_notebooks = [nb for nb in all_notebooks if 'Master_Runner' not in nb and '.ipynb_checkpoints' not in nb]
all_notebooks.sort()

print(f'Found {len(all_notebooks)} notebooks to execute sequentially.')

In [ ]:
# 2. Sequential execution to avoid OOM
results = []
failed = []

for nb_path in tqdm(all_notebooks, desc="Running Models"):
    print(f'\n--- STARTING: {nb_path} ---')
    try:
        # Execute the notebook using subprocess to avoid Marimo conflicts
        subprocess.run(
            [sys.executable, '-m', 'papermill', nb_path, nb_path, '--cwd', paper_dir, '-k', 'python3'],
            check=True
        )
        results.append(nb_path)
        print(f'--- FINISHED: {nb_path} ---')
    except Exception as e:
        print(f'\n[ERROR] Failed to run {nb_path}: {e}')
        failed.append(nb_path)

print(f'\n✅ Execution Complete! Successfully ran {len(results)} out of {len(all_notebooks)} notebooks.')
if failed:
    print(f'Failed notebooks: {len(failed)}')


In [ ]:
# 3. Zip up the results for you to download
# Since you manually uploaded the folder, we will zip it so you can download the evaluated notebooks.
zip_path = '/marimo/Paper_1_Results'
shutil.make_archive(zip_path, 'zip', paper_dir)
print(f'All evaluated notebooks have been zipped to {zip_path}.zip')
print('You can now download this zip file from the Molab UI.')